# 공고 데이터 DB 업데이트
순서대로 실행하세요.

## STEP 1. 패키지 설치

In [ ]:
!pip install xlrd pymysql pandas openpyxl

## STEP 2. 엑셀 파일 읽기 & 컬럼 확인

In [10]:
import pandas as pd

EXCEL_PATH = '/home/jmbyeon/developments/thinkcat-eln/thinkcat-eln/jungmin/공고목록_0619.xls'

df = pd.read_excel(EXCEL_PATH)
print('행 수:', len(df))
print('컬럼 목록:', df.columns.tolist())
df.head(5)

행 수: 1002
컬럼 목록: ['순번', '현황', '형태', '부처명', '공고명', '접수일', '마감일', '접수마감시간', '공고일', '공고기관명', '문의처', '공고문 바로가기(URL)', '공고유형', '공고금액', '공고사업명']


,순번,현황,형태,부처명,공고명,접수일,마감일,접수마감시간,공고일,공고기관명,문의처,공고문 바로가기(URL),공고유형,공고금액,공고사업명
0,76913,접수중,통합공고,해양수산부,2026년도 AI 완전자율운항선박 기술개발사업 신규과제 선정계획 공고_(2026)2...,2026.06.15,2026.07.14,18:00,2026.06.15,해양수산과학기술진흥원,0234600333,https://www.iris.go.kr/contents/retrieveBsnsAn...,본공고,0,AI완전자율운항선박기술개발사업(R&D)
1,76912,접수중,통합공고,보건복지부,2026년도 제3차 첨단재생의료 임상연구 활성화 지원 사업 신규과제 공고_1. [일...,2026.06.10,2026.07.10,16:00,2026.06.10,한국보건산업진흥원,02-6365-2241,https://www.iris.go.kr/contents/retrieveBsnsAn...,본공고,1800000000,첨단재생의료임상연구활성화지원(R&D)
2,76911,접수중,개별공고,우주항공청,2026년도 우주기술혁신인재양성(RD)사업[우주항공 글로벌 인력양성 및 활용] 추가공고,2026.06.10,2026.07.10,18:00,2026.06.10,우주항공청,055-856-5018,https://www.iris.go.kr/contents/retrieveBsnsAn...,본공고,5000000000,우주기술혁신인재양성(R&D)
3,76910,마감,개별공고,과학기술정보통신부,ICT RD 서면검토용 과제접수,2026.06.09,2026.06.09,18:00,2026.06.09,정보통신기획평가원,"042-612-8363,042-612-8366",https://www.iris.go.kr/contents/retrieveBsnsAn...,본공고,0,양자클러스터기획연구
4,76909,마감,개별공고,과학기술정보통신부,(정책지정) 2026년도 양자 ICT RD 정책지정 대상과제 공고,2026.06.09,2026.06.09,18:00,2026.06.09,정보통신기획평가원,"042-612-8363,042-612-8366",https://www.iris.go.kr/contents/retrieveBsnsAn...,본공고,0,양자클러스터기획연구


## STEP 3. 컬럼 매핑 확인
엑셀 컬럼 → DB 컬럼 매핑입니다. 컬럼명이 다르면 여기서 수정하세요.

In [13]:
# 엑셀 컬럼명 : DB 컬럼명
COLUMN_MAP = {
    '공고기관명':             'organization',
    '공고명':                 'title',
    '공고문 바로가기(URL)':   'URL',
    '공고일':                 'announcement_date',
    '접수일':                 'start_date',
    '마감일':                 'end_date',
    '현황':                   'status',
    '공고금액':               'budget',
}

# 매핑 적용
df_mapped = df.rename(columns=COLUMN_MAP)

# DB에 필요한 컬럼만 추출 (없는 컬럼은 None으로)
DB_COLUMNS = ['organization', 'title', 'URL', 'announcement_date', 'start_date', 'end_date', 'status', 'budget']
for col in DB_COLUMNS:
    if col not in df_mapped.columns:
        df_mapped[col] = None

df_mapped = df_mapped[DB_COLUMNS]

# 날짜 컬럼 변환
for date_col in ['announcement_date', 'start_date', 'end_date']:
    df_mapped[date_col] = pd.to_datetime(df_mapped[date_col], errors='coerce').dt.date

# NaN → None
df_mapped = df_mapped.where(pd.notnull(df_mapped), None)

# NOT NULL 컬럼 비어있는 행 제거 & 제거 내용 출력
required_cols = ['organization', 'title', 'URL']
mask_valid = df_mapped[required_cols].notna().all(axis=1) & (df_mapped['organization'].astype(str).str.strip() != '')
dropped = df_mapped[~mask_valid]
if len(dropped) > 0:
    print(f'⚠️ {len(dropped)}행 제거됨:')
    display(dropped)
df_mapped = df_mapped[mask_valid]

print('변환 완료:', len(df_mapped), '행')
df_mapped.head(5)

⚠️ 2행 제거됨:


,organization,title,URL,announcement_date,start_date,end_date,status,budget
1000,NaN,NaN,NaN,None,None,None,NaN,NaN
1001,NaN,NaN,NaN,None,None,None,NaN,NaN


변환 완료: 1000 행


,organization,title,URL,announcement_date,start_date,end_date,status,budget
0,해양수산과학기술진흥원,2026년도 AI 완전자율운항선박 기술개발사업 신규과제 선정계획 공고_(2026)2...,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-15,2026-06-15,2026-07-14,접수중,0
1,한국보건산업진흥원,2026년도 제3차 첨단재생의료 임상연구 활성화 지원 사업 신규과제 공고_1. [일...,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-10,2026-06-10,2026-07-10,접수중,1800000000
2,우주항공청,2026년도 우주기술혁신인재양성(RD)사업[우주항공 글로벌 인력양성 및 활용] 추가공고,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-10,2026-06-10,2026-07-10,접수중,5000000000
3,정보통신기획평가원,ICT RD 서면검토용 과제접수,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-09,2026-06-09,2026-06-09,마감,0
4,정보통신기획평가원,(정책지정) 2026년도 양자 ICT RD 정책지정 대상과제 공고,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-09,2026-06-09,2026-06-09,마감,0


## STEP 4. DB 연결 테스트

In [14]:
import pymysql

DB_CONFIG = {
    'host':     '192.168.1.20',
    'user':     'root',
    'password': 'doslvkdlqm!',
    'database': 'thinkcateln',
    'charset':  'utf8mb4',
}

conn = pymysql.connect(**DB_CONFIG)
cursor = conn.cursor()
cursor.execute('SELECT COUNT(*) FROM announcements')
print('현재 DB 행 수:', cursor.fetchone()[0])
conn.close()
print('연결 성공!')

현재 DB 행 수: 0
연결 성공!


## STEP 5. 기존 데이터 전체 삭제 (테이블 유지)
> ⚠️ 이 셀 실행하면 기존 데이터 전부 삭제됩니다. 확인 후 실행하세요.

In [15]:
conn = pymysql.connect(**DB_CONFIG)
cursor = conn.cursor()
cursor.execute('DELETE FROM announcements')
cursor.execute('ALTER TABLE announcements AUTO_INCREMENT = 1')
conn.commit()
cursor.execute('SELECT COUNT(*) FROM announcements')
print('삭제 후 행 수:', cursor.fetchone()[0])
conn.close()
print('완료!')

KeyboardInterrupt: 

## STEP 6. 엑셀 데이터 DB에 INSERT

In [16]:
from datetime import datetime
import math

def to_none(v):
    try:
        if v is None or (isinstance(v, float) and math.isnan(v)):
            return None
    except Exception:
        pass
    return v

conn = pymysql.connect(**DB_CONFIG)
cursor = conn.cursor()

sql = """
INSERT INTO announcements
    (organization, title, URL, announcement_date, start_date, end_date, status, budget, created_at, updated_at)
VALUES
    (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

now = datetime.now()
rows = []
for _, row in df_mapped.iterrows():
    rows.append((
        to_none(row['organization']),
        to_none(row['title']),
        to_none(row['URL']),
        to_none(row['announcement_date']),
        to_none(row['start_date']),
        to_none(row['end_date']),
        to_none(row['status']),
        to_none(row['budget']),
        now,
        now,
    ))

cursor.executemany(sql, rows)
conn.commit()
print(f'INSERT 완료: {cursor.rowcount}행')
conn.close()

INSERT 완료: 1000행


## STEP 7. 결과 확인

In [17]:
conn = pymysql.connect(**DB_CONFIG)
df_check = pd.read_sql('SELECT * FROM announcements ORDER BY id DESC LIMIT 10', conn)
conn.close()
print('최종 DB 행 수:', len(pd.read_sql('SELECT id FROM announcements', pymysql.connect(**DB_CONFIG))))
df_check

최종 DB 행 수: 1000


/tmp/ipykernel_2295866/387620701.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_check = pd.read_sql('SELECT * FROM announcements ORDER BY id DESC LIMIT 10', conn)
/tmp/ipykernel_2295866/387620701.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print('최종 DB 행 수:', len(pd.read_sql('SELECT id FROM announcements', pymysql.connect(**DB_CONFIG))))


,id,organization,title,announcement_date,start_date,end_date,status,URL,created_at,updated_at,budget
0,2002,국토교통과학기술진흥원,(공고-국-제08호) 2026년 철도차량 차륜 자동검사시스템 기술개발 사업 시행 공...,2026-01-08,2026-01-20,2026-02-09,마감,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-19 16:33:10,2026-06-19 16:33:10,0
1,2001,국토교통과학기술진흥원,(공고-국-제09호) 2026년 빅데이터 기반 철도 네트워크 설계 및 운영기술 개발...,2026-01-08,2026-01-20,2026-02-09,마감,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-19 16:33:10,2026-06-19 16:33:10,0
2,2000,중소기업기술정보진흥원,2026년도 딥테크 챌린지 프로젝트(DCP)생태계혁신형 시행계획 공고_(2026)투...,2026-01-08,2026-02-20,2026-04-07,마감,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-19 16:33:10,2026-06-19 16:33:10,20000000000
3,1999,국토교통과학기술진흥원,(공고-국-제19호) 2026년 액체수소 저장탱크 및 적하역 시스템 기술개발 사업 ...,2026-01-09,2026-01-19,2026-02-09,마감,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-19 16:33:10,2026-06-19 16:33:10,29000000000
4,1998,과학치안진흥센터,2026년도 미래치안 도전기술 개발(핵심원천) 신규과제 공모_(2026)2026년도...,2026-01-09,2026-01-26,2026-02-09,마감,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-19 16:33:10,2026-06-19 16:33:10,1500000000
5,1997,국립암센터,2026년도 면역세포유전자치료제전주기기술개발사업 연구지원과제 공고 _(2026)면역...,2026-01-09,2026-01-09,2026-02-09,마감,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-19 16:33:10,2026-06-19 16:33:10,69362000000
6,1996,농림식품기술기획평가원,2026년도 고부가가치식품기술개발사업 시행계획 공고,2026-01-09,2026-01-16,2026-02-09,마감,http://www.ipet.re.kr/Rnd/bizNoticeVP.asp?page...,2026-06-19 16:33:10,2026-06-19 16:33:10,11500000000
7,1995,농림식품기술기획평가원,2026년도 고부가가치식품기술개발사업 시행계획 공고_(2026)2026년도 고부가가...,2026-01-09,2026-01-19,2026-02-09,마감,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-19 16:33:10,2026-06-19 16:33:10,33816000000
8,1994,농림식품기술기획평가원,2026년도 고부가가치식품기술개발사업 시행계획 공고_(2026)2026년도 고부가가...,2026-01-09,2026-01-19,2026-02-09,마감,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-19 16:33:10,2026-06-19 16:33:10,33816000000
9,1993,농림식품기술기획평가원,2026년도 고부가가치식품기술개발사업 시행계획 공고_(2026)2026년도 고부가가...,2026-01-09,2026-01-19,2026-02-09,마감,https://www.iris.go.kr/contents/retrieveBsnsAn...,2026-06-19 16:33:10,2026-06-19 16:33:10,33816000000
